### Implementação de Fuzzy C-means utilizando GloVe

In [2]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px
import skfuzzy as fuzz
from plotly.subplots import make_subplots


In [62]:
def load_glove(file_path):
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

def words_to_vectors(words, embeddings, dimension=50):
    vectors = []
    for word in words:
        vector = embeddings.get(word)
        if vector is not None:
            vectors.append(vector)
        else:
            print(f"'{word}' not found in GloVe vocabulary. Using zero vector.")
            vectors.append(np.zeros(dimension))
    return np.array(vectors)

glove_file_path = './GloVe/glove.6B.50d.txt'

glove_embeddings = load_glove(glove_file_path)


In [63]:
words = ['harvard', 'learning', 'intelligence']

word_vectors = words_to_vectors(words, glove_embeddings)
print(word_vectors)

[[-8.5970e-01  1.1120e+00 -2.9970e-01 -1.1093e+00  1.5653e-01 -1.3244e-01
  -1.0520e+00 -9.2620e-01 -5.2920e-01 -2.4501e-01 -2.2653e-01  2.5299e-01
  -9.9125e-02 -4.0640e-01  9.7853e-04 -3.5808e-02 -1.8689e-01  7.1157e-01
  -4.4480e-01  8.6651e-01  5.4339e-01  5.9826e-01 -3.1584e-02 -4.6351e-01
  -8.5038e-02 -1.8902e+00  1.1140e-01 -7.5604e-01 -1.6965e+00 -3.9752e-01
   1.2976e+00 -3.4127e-01 -2.2890e-01 -1.4524e+00 -2.9855e-01 -2.0297e-01
  -4.4211e-01  1.1521e+00  1.5059e+00 -4.8819e-01 -2.1176e-01 -3.6186e-01
  -9.1108e-02  9.5266e-01  2.0254e-01  1.0068e-01  6.9316e-01  2.6215e-01
  -9.0986e-01  5.9507e-01]
 [ 2.0461e-01  4.8659e-01 -5.5308e-01 -2.7019e-01  2.6336e-01  1.5751e-01
  -2.8994e-01 -5.1824e-01  5.1829e-02  3.6225e-01  3.7077e-01  1.3220e-01
  -6.1377e-02 -5.3606e-01 -3.4733e-01 -4.3981e-02 -8.6744e-02  7.8305e-01
   4.1422e-01  2.7996e-02  2.3433e-01  9.8844e-01 -4.1049e-01  6.2060e-01
   1.3966e+00 -6.5427e-01 -1.8221e-01 -1.0293e+00 -1.4741e-02 -2.5384e-01
   3.2270e+

In [64]:
wordsim_path_file = './WordSim_353/wordsim_relatedness_goldstandard.txt'
df = pd.read_csv(wordsim_path_file, sep='\t', header=None)
print(df.head())
words = pd.concat([df[0], df[1]]).str.lower().drop_duplicates()
word_vectors = words_to_vectors(words, glove_embeddings)
print('#words = ', len(words))

           0          1     2
0   computer   keyboard  7.62
1  Jerusalem     Israel  8.46
2     planet     galaxy  8.11
3     canyon  landscape  7.53
4       OPEC    country  5.63
#words =  346


In [65]:
# perplexity = np.arange(10, 300, 10)
# divergence = []

# for i in perplexity:
#     model = TSNE(n_components=2, init="pca", perplexity=i)
#     reduced = model.fit_transform(word_vectors)
#     divergence.append(model.kl_divergence_)
# fig = px.line(x=perplexity, y=divergence, markers=True)
# fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="Divergence")
# fig.update_traces(line_color="red", line_width=1)
# fig.show()

In [66]:
tsne = TSNE(n_components=2,perplexity=20, init='pca', random_state=0)
word_vectors_tsne = tsne.fit_transform(word_vectors)

tsne.kl_divergence_

1.2116248607635498

In [67]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=800,
    height=500
)
fig.show()

In [68]:
n_clusters = 8
fuzz_par = 1.1

fcm_model = fuzz.cluster.cmeans(word_vectors.T, c=n_clusters, m=fuzz_par, error=0.05, maxiter=1000, init=None)
cluster_membership = np.argmax(fcm_model[1], axis=0)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

top_words_per_cluster = {}

top_n = 100

for i in range(n_clusters):
    # Seleciona as top_n palavras ordenadas pelo grau de pertencimento ao cluster i
    top_entries = df_membership[f"Cluster {i}"].sort_values(ascending=False).head(top_n)
    top_words_per_cluster[f"Cluster {i}"] = [f"{word} ({value:.4f})" for word, value in zip(top_entries.index, top_entries.values)]

df_top_words = pd.DataFrame.from_dict(top_words_per_cluster, orient="index")

display(df_top_words)

df_membership = pd.DataFrame(fcm_model[1].T, index=words, columns=[f"Cluster {i}" for i in range(n_clusters)])

df_membership = df_membership.map(lambda x: f"{x:.4f}")

display(df_membership)

fcm_im = fcm_model[1] @ fcm_model[1].T

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
Cluster 0,government (0.9992),plan (0.9970),possibility (0.9941),withdrawal (0.9939),issue (0.9934),delay (0.9916),israel (0.9904),ministry (0.9854),planning (0.9852),effort (0.9840),...,holy (0.0679),car (0.0614),street (0.0602),admission (0.0564),string (0.0542),weather (0.0530),ticket (0.0527),accommodation (0.0508),tiger (0.0506),treatment (0.0505)
Cluster 1,nature (0.9942),morality (0.9895),isolation (0.9824),cognition (0.9814),psychology (0.9750),prejudice (0.9745),gender (0.9737),experience (0.9650),importance (0.9613),attitude (0.9611),...,word (0.1423),marriage (0.1379),student (0.1344),recommendation (0.1323),scientist (0.1322),exhibit (0.1199),maradona (0.1145),century (0.1143),registration (0.1084),virtuoso (0.1077)
Cluster 2,lover (0.9992),brother (0.9979),doctor (0.9978),man (0.9976),mother (0.9972),girl (0.9891),monk (0.9861),love (0.9844),life (0.9841),victim (0.9814),...,collection (0.0609),physics (0.0608),governor (0.0604),chemistry (0.0603),practice (0.0600),closet (0.0593),zoo (0.0577),alcohol (0.0577),rooster (0.0576),hundred (0.0573)
Cluster 3,round (1.0000),match (0.9999),team (0.9996),season (0.9994),game (0.9989),cup (0.9976),record (0.9972),competition (0.9970),victory (0.9965),football (0.9942),...,constellation (0.0146),board (0.0146),senate (0.0142),journal (0.0140),king (0.0139),confidence (0.0136),flight (0.0135),smile (0.0135),war (0.0132),attempt (0.0132)
Cluster 4,computer (0.9978),hardware (0.9970),internet (0.9963),software (0.9963),video (0.9932),phone (0.9911),network (0.9898),media (0.9803),information (0.9737),telephone (0.9645),...,school (0.0652),constellation (0.0620),grocery (0.0611),center (0.0584),clinic (0.0580),exhibit (0.0569),property (0.0544),industry (0.0520),preservation (0.0505),culture (0.0493)
Cluster 5,shore (0.9972),area (0.9907),canyon (0.9902),sea (0.9866),mars (0.9794),proximity (0.9668),coast (0.9622),forest (0.9618),observation (0.9567),graveyard (0.9547),...,development (0.0737),mouth (0.0735),slave (0.0703),opera (0.0686),library (0.0655),string (0.0573),arrangement (0.0569),board (0.0512),possession (0.0497),environment (0.0492)
Cluster 6,price (1.0000),stock (1.0000),market (1.0000),profit (0.9998),trading (0.9996),dollar (0.9995),interest (0.9995),investor (0.9988),credit (0.9986),currency (0.9982),...,governor (0.0069),depression (0.0065),admission (0.0064),journal (0.0063),senate (0.0062),string (0.0058),fertility (0.0056),territory (0.0055),criterion (0.0054),project (0.0052)
Cluster 7,popcorn (0.9995),drink (0.9992),coffee (0.9983),egg (0.9967),sugar (0.9955),cabbage (0.9952),shower (0.9949),seafood (0.9920),liquid (0.9862),cucumber (0.9854),...,hundred (0.0169),wealth (0.0160),collection (0.0157),word (0.0153),graveyard (0.0149),depression (0.0147),galaxy (0.0146),music (0.0145),reservation (0.0141),smart (0.0140)


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7
computer,0.0001,0.0007,0.0005,0.0001,0.9978,0.0002,0.0005,0.0001
jerusalem,0.6545,0.0384,0.0562,0.0042,0.0130,0.2280,0.0028,0.0029
planet,0.0017,0.0124,0.0176,0.0018,0.0144,0.9437,0.0005,0.0080
canyon,0.0007,0.0014,0.0023,0.0006,0.0018,0.9902,0.0003,0.0028
opec,0.2438,0.0203,0.0060,0.0125,0.0102,0.0133,0.6322,0.0617
...,...,...,...,...,...,...,...,...
voyage,0.0074,0.0072,0.0373,0.0097,0.0046,0.9288,0.0006,0.0043
string,0.0542,0.0987,0.1436,0.4477,0.1619,0.0573,0.0058,0.0307
smile,0.0060,0.0647,0.3899,0.0135,0.0281,0.0345,0.0016,0.4618
cucumber,0.0005,0.0014,0.0027,0.0019,0.0017,0.0057,0.0006,0.9854


In [69]:
fig = px.scatter(x=word_vectors_tsne[:, 0], y=word_vectors_tsne[:, 1], text=words, color=cluster_membership)
fig.update_layout(
    title="t-SNE visualization of WordSim_353 dataset with FCM",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
    width=1500,
    height=900
)
fig.show()

fcm_im = fcm_model[1].T @ fcm_model[1]

df_membership.index = np.array(df_membership.index)
df_membership['cluster'] = cluster_membership

df_membership_ord = df_membership.sort_values(by='cluster')
df_membership_ord = df_membership_ord.drop(columns='cluster')
# display(df_membership_ord)

df_membership_ord = np.array(df_membership_ord).astype(float)
fcm_im_ord = df_membership_ord @ df_membership_ord.T

# Create a subplot with two columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Affinity matrix", "Ordered affinity matrix"),
    column_widths=[0.5, 0.5]
)

# Add the first heatmap
fig.add_trace(
    px.imshow(fcm_im, color_continuous_scale='plasma').data[0],
    row=1, col=1
)

# Add the second heatmap
fig.add_trace(
    px.imshow(fcm_im_ord, color_continuous_scale='plasma').data[0],
    row=1, col=2
)

# Update layout
fig.update_layout(
    title="Comparison of FCM Membership Matrices",
    width=1200,
    height=600
)

fig.show()

In [73]:
# Encontre as palavras que tem maior valor de pertinência menor que 0.5
words_below_threshold = []
for i in range(words.shape[0]):
    words_below_threshold.append(float(df_membership.iloc[i,cluster_membership[i]]) < 0.4)

# Exiba as palavras que têm maior valor de pertinência menor que 0.5
display(df_membership[words_below_threshold])

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,cluster
fbi,0.3672,0.0518,0.1962,0.0049,0.2978,0.0749,0.0042,0.0030,0
drug,0.1608,0.2724,0.1883,0.0288,0.2017,0.0257,0.0674,0.0550,1
concert,0.0244,0.0202,0.2882,0.2674,0.1649,0.1939,0.0027,0.0382,2
luxury,0.0229,0.0345,0.0401,0.0240,0.2331,0.3060,0.1507,0.1886,5
weapon,0.1186,0.2388,0.1239,0.0291,0.3678,0.0800,0.0041,0.0378,4
decoration,0.0174,0.3958,0.1033,0.0195,0.1033,0.2024,0.0045,0.1539,1
arrangement,0.2199,0.2579,0.0339,0.0050,0.3920,0.0569,0.0217,0.0126,4
size,0.0107,0.1029,0.0152,0.0209,0.1736,0.2078,0.1255,0.3435,7
development,0.3487,0.3860,0.0053,0.0076,0.1154,0.0737,0.0623,0.0009,1
population,0.1325,0.3315,0.0538,0.0183,0.0206,0.3807,0.0524,0.0101,5


### Critérios de avaliação de Clusters

#### Internal Clustering Indices
- Avalia a qualidade de uma clusterização sem usar rótulos/infomrações externas;
- Baseia-se somente na estrutura intrínseca dos dados;

- **Silhouette Coefficient**: mede a similaridade de um ponto com o seu próprio _cluster_ em comparação com outros _clusters_. O valor varia entre -1 e 1, onde valores próximos a 1 indicam que o ponto está bem posicionado no seu _cluster_ e longe dos outros. Valores próximos a 0 indicam que o ponto está próximo da borda entre dois _clusters_.
    $$ s_i = \frac{b_i-a_i}{\max(a_i,b_i)} $$
    onde:
    - $a$ é a distância média entre o ponto e os pontos do seu próprio _cluster_;
    - $b$ é a menor distância média entre o ponto e todos os outros _clusters_;

In [ ]:
def silhouette_score(Xy, n_clusters):
    n_samples = Xy.shape[0]
    for sample_i in Xy:
        a = 0
        b = 0
        n_samples_c = 0
        for cluster in range(n_clusters):
            cluster_samples = np.where(Xy[-1] == cluster)
            if cluster == sample_i[-1]:
                a = np.sum([np.linalg.norm(sample_i[:-1], Xy[cluster_samples][j][:-1]) for j in range(len(cluster_samples))])
        for sample_j in Xy:
            if sample_j[-1] == sample_i[-1]:
                a += np.linalg.norm(sample_i[:-1], sample_j[:-1])
                n_samples_c +=1
            else:

In [42]:
matriz = np.random.uniform(0, 10, (4, 3))
print(matriz)
cluster_samples = np.where(matriz[:,-1] > 7)
print(cluster_samples)
a = [np.linalg.norm(matriz[0, :-1] - matriz[idx, :-1]) for idx in cluster_samples[0]]
print(a)
print(np.sum(a))

[[4.97039103 5.98044816 1.63199752]
 [0.86735841 7.4474818  2.49810431]
 [4.55675731 5.08668373 5.48094326]
 [2.51254008 8.62611564 3.84260007]]
(array([], dtype=int64),)
[]
0.0


In [34]:
print(matriz[1,:-1])
print(matriz[cluster_samples][0,:-1])
print(len(cluster_samples))

print(np.linalg.norm(matriz[1, :-1] - matriz[cluster_samples[0][0], :-1]))

[0.03420629 6.588972  ]
[8.74391395 8.69964377]
1
8.961804662382868


- **Calinski-Harabasz Index ('Critério da Razão de Variância')**: maior pontuação significa que os _clusters_ são densos e bem separados. Entretanto é mais alto para _clusters_ convexos do que ára outros conceitos de _clusters_ como os baseados em densidade.
    $$ \begin{align} s &= \frac{tr(B_k)}{tr(W_k)} \times \frac{n-k}{k-1} \\ W_k&=\sum^k_{q=1} \sum _{x \in C_q} (x-c_q)(x-c_q)^T \\ B_k &= \sum_{q=1}^k n_q(c_q-c_E)(c_q-c_e)^T \end{align} $$
    onde:
    - $tr(x)$ é o traço de x;
    - $B_k$ é a disperção entre grupos;
    - $W_k$ é a disperção entre _clusters_;
    - $C_q$ são os pontos do _cluster_ $q$;
    - $c_q$ é o centro do _cluster_ $q$;
    - $c_E$ é o centro de E;
    - $n_q$ é o número de pontos no _cluster_ $q$
    - $k$ é o número de _clusters_;
    - $n_E$ é o número total de pontos;
- **Davies-Bouldin Index**: mede a separação entre os _clusters_ e a sua compactação através de um cálculo de 'similaridade'. Quanto menor o índice, melhor a separação entre os _clusters_.
    $$ DB = \frac{1}{k} \sum_{i=1}^k \max_{j \neq i} \left( \frac{s_i + s_j}{d(c_i, c_j)} \right) $$
    onde:
    - $s_i$ é a dispersão do _cluster_ $i$ (distância média entre os pontos do _cluster_ e o centro do _cluster_);
    - $d(c_i, c_j)$ é a distância entre os centros dos _clusters_ $i$ e $j$;
    - $k$ é o número de _clusters_.
- **Dunn Index**: mede a separação entre os _clusters_ e a sua compactação através de um cálculo de 'similaridade'. Quanto maior o índice, melhor a separação entre os _clusters_.
    $$ D = \frac{1}{k} \sum_{i=1}^k \max_{j \neq i} \left( \frac{s_i + s_j}{d(c_i, c_j)} \right) $$
    onde:
    - $s_i$ é a dispersão do _cluster_ $i$ (distância média entre os pontos do _cluster_ e o centro do _cluster_);
    - $d(c_i, c_j)$ é a distância entre os centros dos _clusters_ $i$ e $j$;
    - $k$ é o número de _clusters_.

#### External Clustering Indices
- Avalia a qualidade de uma clusterização utilizando rótulos/infomrações externas;
- Requer conhecimento prévio dos rótulos dos dados;
- Exemplos:
    - **Rand Index**: mede a similaridade entre dois _clusters_ (ou partições) comparando os pares de pontos. O valor varia entre 0 e 1, onde 1 indica que os dois _clusters_ são idênticos e 0 indica que não há similaridade.
        $$ RI = \frac{TP + TN}{TP + TN + FP + FN} $$
        onde:
        - $TP$ é o número de pares de pontos que estão no mesmo _cluster_ em ambas as partições;
        - $TN$ é o número de pares de pontos que estão em _clusters_ diferentes em ambas as partições;
        - $FP$ é o número de pares de pontos que estão no mesmo _cluster_ na primeira partição, mas em _clusters_ diferentes na segunda;
        - $FN$ é o número de pares de pontos que estão em _clusters_ diferentes na primeira partição, mas no mesmo _cluster_ na segunda.
    - **Adjusted Rand Index**: versão ajustada do Rand Index que leva em conta a chance de coincidência aleatória. O valor varia entre -1 e 1, onde 1 indica que os dois _clusters_ são idênticos, 0 indica que a similaridade é igual à chance aleatória e -1 indica que os dois _clusters_ são completamente diferentes.
        $$ ARI = \frac{RI - E[RI]}{\max(RI) - E[RI]} $$
        onde:
        - $E[RI]$ é a expectativa do Rand Index sob a hipótese nula de que as duas partições são independentes.
    - **Fowlkes-Mallows Index**: mede a similaridade entre dois _clusters_ (ou partições) comparando os pares de pontos. O valor varia entre 0 e 1, onde 1 indica que os dois _clusters_ são idênticos e 0 indica que não há similaridade.
        $$ FM = \frac{TP}{\sqrt{(TP + FP)(TP + FN)}} $$